# 04 Intervention ROI Simulation

## H4 Definition
**H4:** Targeted intervention on high-risk seller-days can reduce future severe-event harm with positive ROI, while keeping GMV-side guardrails within acceptable bounds.

## Objectives
- Rebuild the default H3 deployment ranking:
  - Logistic Regression
  - Horizon = 14 days
  - Top-K intervention policy inherited from H3
- Explicitly define the **no-intervention baseline**:
  - future severe-event count
  - future severe-event GMV
  - harm proxy in BRL
- Translate H2 customer-harm evidence into business proxies:
  - avoided incremental low ratings (monetised proxy)
  - avoided review-score loss (non-monetised KPI)
- Simulate multiple intervention scenarios under:
  - conservative
  - base
  - aggressive
  assumption profiles
- Compare:
  - captured future severe-event GMV
  - prevented future severe-event GMV
  - intervention cost
  - guardrail impact
  - net benefit and ROI
- Run explicit **K-sensitivity analysis** for top-K intervention thresholds.

## Guardrails
In this notebook, guardrail impact is defined as:
- `current_gmv_proxy_share`: share of current 14-day GMV footprint affected by intervention
- `throttled_gmv_brl`: GMV directly suppressed under restrictive actions
- `margin_loss_brl`: contribution-margin loss implied by throttled GMV

## Notes
- This is a scenario-based ROI simulation, not a causal policy estimate.
- Because H3 calibration is not yet fully validated, H4 uses a **ranked top-K intervention framework**
  rather than probability-weighted expected loss.
- Default K = 5% seller-days, inherited from H3 deployment recommendation.
- K sensitivity is evaluated at K {1%, 3%, 5%, 10%}.

## 0. Import & Settings

In [4]:
# %%
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import altair as alt
from IPython.core.interactiveshell import InteractiveShell
from pathlib import Path
import sys

pd.set_option("display.max_columns", 80)
InteractiveShell.ast_node_interactivity = "all"
alt.data_transformers.disable_max_rows()

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from config import DATA_INTERIM, DATA_PROCESSED
from data.preprocessing import load_orders_sellers

from features.intervention_roi import (
    build_feature_list,
    ensure_trend_features,
    time_split_and_trim,
    derive_h2_harm_coefficients,
    compute_no_intervention_baseline,
    build_assumption_profiles,
)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


DataTransformerRegistry.enable('default')

## 1. Load Inputs

We load the key artifacts from prior modules:
- `orders_sellers` from 00 (SSoT with SLA + GMV)
- `cx_dose_response.parquet` from 02 (H2 customer-harm curve)
- `seller_early_warning_panel.parquet` from 03 (H3 feature panel + future labels)

In [5]:
orders_sellers = load_orders_sellers()
dose_summary = pd.read_parquet(DATA_INTERIM / "cx_dose_response.parquet")
early_panel = pd.read_parquet(DATA_INTERIM / "seller_early_warning_panel.parquet")

print(f"orders_sellers shape: {orders_sellers.shape}")
print(f"dose_summary shape: {dose_summary.shape}")
print(f"early_panel shape: {early_panel.shape}")

orders_sellers.head(3)
dose_summary
early_panel.head(3)

orders_sellers shape: (99441, 13)
dose_summary shape: (4, 6)
early_panel shape: (42644, 71)


,order_id,seller_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delay_days,is_sla_violation,is_severe_violation,has_time_anomaly,product_category_name_english,order_gmv
0,e481f51cbdc54678b7cc49136f2d6af7,3504c0cb71d7fa48d967e0e4c94d59d9,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,-8.0,False,False,False,housewares,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,289cdb325fb7e7f891c38608bf9e0962,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,-6.0,False,False,False,perfumery,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,4869f7a5dfa277a7dca6462dcf3b52b2,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,-18.0,False,False,False,auto,179.12


,delay_bucket,orders,mean_review,low_rating_rate,cancel_rate,repeat_rate
0,on_time_or_early,89941,4.289842,0.172641,0.000055,0.005539
1,1-2_days_late,1370,3.511029,0.403202,0.000000,0.002911
2,3-5_days_late,1400,2.467495,0.669280,0.000000,0.002138
3,6+_days_late,3765,1.740016,0.846540,0.000264,0.005547


,date,seller_id,delivered_orders,sla_violations,severe_violations,avg_delay_days,delivered_gmv,violation_gmv,severe_violation_gmv,violation_rate,severe_violation_rate,violation_gmv_share,severe_violation_gmv_share,first_order_date,seller_tenure_days,cum_delivered_orders,cum_sla_violations,cum_severe_violations,cum_delivered_gmv,cum_violation_gmv,cum_severe_violation_gmv,lifetime_violation_rate,lifetime_severe_violation_rate,lifetime_violation_gmv_share,lifetime_severe_violation_gmv_share,delivered_7d,sla_violations_7d,severe_violations_7d,delivered_gmv_7d,violation_gmv_7d,severe_violation_gmv_7d,violation_rate_7d,severe_violation_rate_7d,violation_gmv_share_7d,severe_violation_gmv_share_7d,avg_delay_7d,delivered_14d,sla_violations_14d,severe_violations_14d,delivered_gmv_14d,violation_gmv_14d,severe_violation_gmv_14d,violation_rate_14d,severe_violation_rate_14d,violation_gmv_share_14d,severe_violation_gmv_share_14d,avg_delay_14d,delivered_30d,sla_violations_30d,severe_violations_30d,delivered_gmv_30d,violation_gmv_30d,severe_violation_gmv_30d,violation_rate_30d,severe_violation_rate_30d,violation_gmv_share_30d,severe_violation_gmv_share_30d,avg_delay_30d,violation_rate_trend_7v30,severe_rate_trend_7v30,gmv_share_trend_7v30,delay_trend_7v30,future_severe_count_7d,future_severe_gmv_7d,label_future_severe_7d,future_severe_count_14d,future_severe_gmv_14d,label_future_severe_14d,future_severe_count_21d,future_severe_gmv_21d,label_future_severe_21d
0,2017-04-10,001cca7ae9ae17fb1caed9dfb1094831,1,0,0,-22.0,129.06,0.00,0.00,0.0,0.0,0.0,0.0,2017-02-04,66,20,1,1,3903.49,109.83,109.83,0.050000,0.050000,0.028136,0.028136,3.0,0.0,0.0,382.96,0.00,0.00,0.00,0.00,0.000000,0.000000,-17.666667,6.0,0.0,0.0,741.80,0.00,0.00,0.000000,0.000000,0.000000,0.000000,-17.333333,13.0,1.0,1.0,2251.22,109.83,109.83,0.076923,0.076923,0.048787,0.048787,-12.954545,-0.076923,-0.076923,-0.048787,-4.712121,1,120.36,1,1,120.36,1,1,120.36,1
1,2017-04-11,001cca7ae9ae17fb1caed9dfb1094831,2,0,0,-19.5,385.44,0.00,0.00,0.0,0.0,0.0,0.0,2017-02-04,67,22,1,1,4288.93,109.83,109.83,0.045455,0.045455,0.025608,0.025608,5.0,0.0,0.0,768.40,0.00,0.00,0.00,0.00,0.000000,0.000000,-18.125000,7.0,0.0,0.0,970.66,0.00,0.00,0.000000,0.000000,0.000000,0.000000,-17.083333,15.0,1.0,1.0,2636.66,109.83,109.83,0.066667,0.066667,0.041655,0.041655,-13.500000,-0.066667,-0.066667,-0.041655,-4.625000,1,120.36,1,1,120.36,1,1,120.36,1
2,2017-04-16,001cca7ae9ae17fb1caed9dfb1094831,1,1,1,16.0,120.36,120.36,120.36,1.0,1.0,1.0,1.0,2017-02-04,72,23,2,2,4409.29,230.19,230.19,0.086957,0.086957,0.052206,0.052206,4.0,1.0,1.0,634.86,120.36,120.36,0.25,0.25,0.189585,0.189585,-8.500000,6.0,1.0,1.0,888.76,120.36,120.36,0.166667,0.166667,0.135425,0.135425,-11.300000,14.0,2.0,2.0,2244.66,230.19,230.19,0.142857,0.142857,0.102550,0.102550,-11.818182,0.107143,0.107143,0.087035,3.318182,0,0.00,0,0,0.00,0,0,0.00,0


## 2. Derive H2 Harm Coefficients
H2 established dose–response between delay severity and customer harm.
For H4, we need an operational bridge from H2 to economic simulation.

We compare:
- `3-5_days_late` vs `on_time_or_early` → moderate-harm coefficients
- `6+_days_late`  vs `on_time_or_early` → severe-harm coefficients

The main ROI simulation uses the **severe** coefficients because H3 predicts future severe SLA events.

In [6]:
harm_table = derive_h2_harm_coefficients(
    dose_summary=dose_summary,
    reference_bucket="on_time_or_early",
    moderate_bucket="3-5_days_late",
    severe_bucket="6+_days_late",
)

harm_table

,bucket_type,bucket,reference_bucket,ref_mean_review,ref_low_rating_rate,ref_cancel_rate,ref_repeat_rate,bucket_mean_review,bucket_low_rating_rate,bucket_cancel_rate,bucket_repeat_rate,delta_review_loss,delta_low_rating,delta_cancel_rate,delta_repeat_rate
0,moderate,3-5_days_late,on_time_or_early,4.289842,0.172641,0.000055,0.005539,2.467495,0.66928,0.000000,0.002138,1.822348,0.496639,-0.000055,0.003401
1,severe,6+_days_late,on_time_or_early,4.289842,0.172641,0.000055,0.005539,1.740016,0.84654,0.000264,0.005547,2.549826,0.673899,0.000209,-0.000007


In [7]:
moderate_harm = harm_table.loc[harm_table["bucket_type"] == "moderate"].iloc[0].to_dict()
severe_harm = harm_table.loc[harm_table["bucket_type"] == "severe"].iloc[0].to_dict()

print("Moderate-harm coefficients")
pd.Series(moderate_harm)

print("\nSevere-harm coefficients")
pd.Series(severe_harm)

Moderate-harm coefficients


bucket_type                       moderate
bucket                       3-5_days_late
reference_bucket          on_time_or_early
ref_mean_review                   4.289842
ref_low_rating_rate               0.172641
ref_cancel_rate                   0.000055
ref_repeat_rate                   0.005539
bucket_mean_review                2.467495
bucket_low_rating_rate             0.66928
bucket_cancel_rate                     0.0
bucket_repeat_rate                0.002138
delta_review_loss                 1.822348
delta_low_rating                  0.496639
delta_cancel_rate                -0.000055
delta_repeat_rate                 0.003401
dtype: object


Severe-harm coefficients


bucket_type                         severe
bucket                        6+_days_late
reference_bucket          on_time_or_early
ref_mean_review                   4.289842
ref_low_rating_rate               0.172641
ref_cancel_rate                   0.000055
ref_repeat_rate                   0.005539
bucket_mean_review                1.740016
bucket_low_rating_rate             0.84654
bucket_cancel_rate                0.000264
bucket_repeat_rate                0.005547
delta_review_loss                 2.549826
delta_low_rating                  0.673899
delta_cancel_rate                 0.000209
delta_repeat_rate                -0.000007
dtype: object

## 3. Rebuild the Default H3 Deployment Score

H3 recommended:
- model = Logistic Regression
- horizon = 14 days
- default deployment policy = top 5% seller-days

We rebuild the exact scoring pipeline here so that H4 can simulate ranked interventions at the seller-day level.

In [8]:
early_panel = ensure_trend_features(early_panel)
windows = [7, 14, 30]
horizons = [7, 14, 21]
max_horizon = max(horizons)

feature_cols = build_feature_list(windows=windows, include_trends=True)

train_df_full, test_df_full, train_end_date = time_split_and_trim(
    df=early_panel,
    date_col="date",
    train_frac=0.70,
    max_horizon_days=max_horizon,
)

print(f"Train rows: {len(train_df_full):,}")
print(f"Test rows: {len(test_df_full):,}")
print(f"Raw train end date: {train_end_date}")

Train rows: 21,796
Test rows: 16,497
Raw train end date: 2018-03-06T00:00:00.000000000


In [9]:
# Default H3 deployment horizon
horizon = 14
label_col = f"label_future_severe_{horizon}d"
future_event_count_col = f"future_severe_count_{horizon}d"
future_event_gmv_col = f"future_severe_gmv_{horizon}d"

X_train = train_df_full[feature_cols].fillna(0.0).values
y_train = train_df_full[label_col].astype(int).values

X_test = test_df_full[feature_cols].fillna(0.0).values
y_test = test_df_full[label_col].astype(int).values

lr_14d = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
)
lr_14d.fit(X_train, y_train)

test_scored = test_df_full.copy()
test_scored["score_lr_14d"] = lr_14d.predict_proba(X_test)[:, 1]

test_scored[[
    "seller_id", "date", "score_lr_14d",
    label_col, future_event_count_col, future_event_gmv_col
]].head()

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

,seller_id,date,score_lr_14d,label_future_severe_14d,future_severe_count_14d,future_severe_gmv_14d
25735,8d956fec2e4337affcb520f56fd8cbfd,2018-03-07,0.461790,0,0,0.00
2673,0ffa40d54288e4f3499b8780dd0f144f,2018-03-07,0.408878,0,0,0.00
40072,f262cbc1c910c83959f849465454ddd3,2018-03-07,0.471656,1,1,45.22
24413,870d0118f7a9d85960f29ad89d5d989a,2018-03-07,0.386227,0,0,0.00
34798,d20b021d3efdf267a402c402a48ea64b,2018-03-07,0.630300,0,0,0.00


In [10]:
# Sanity check: score quality should be aligned with H3
auc_14d = roc_auc_score(y_test, test_scored["score_lr_14d"])
ap_14d = average_precision_score(y_test, test_scored["score_lr_14d"])

print(f"H=14d Logistic Regression test ROC AUC: {auc_14d:.4f}")
print(f"H=14d Logistic Regression test Average Precision: {ap_14d:.4f}")

H=14d Logistic Regression test ROC AUC: 0.7529
H=14d Logistic Regression test Average Precision: 0.3826


## 4. Explicit No-Intervention Baseline

This section formalises the benchmark that all H4 scenarios will be compared against.

Economic proxy design:
- **Compensation cost proxy** = future severe-event GMV × compensation_rate_on_prevented_gmv
- **Reputation cost proxy** = incremental low ratings × cost_per_incremental_low_rating_proxy_brl
- **Review-score loss** is retained as a non-monetised supporting KPI (not included in ROI)

This baseline answers:
> If we do nothing, how much future severe-event harm exists in the deployment window?

In [12]:
assumption_profiles = build_assumption_profiles()

baseline_df = compute_no_intervention_baseline(
    scored_df = test_scored,
    future_event_count_col = future_event_count_col,
    future_event_gmv_col = future_event_gmv_col,
    severe_harm_row = severe_harm,
    assumption_profiles = assumption_profiles,
    current_gmv_proxy_col = "delivered_gmv_14d"
)

baseline_df

,assumption_profile,seller_days,unique_sellers,total_future_events,total_future_gmv_brl,incremental_low_ratings_proxy,review_points_lost,compensation_cost_proxy_brl,reputation_cost_proxy_brl,total_harm_proxy_brl,current_gmv_proxy_brl
0,conservative,16497,698,3608,551465.54,2431.427066,9199.771204,5514.6554,14588.562395,20103.217795,27645323.52
1,base,16497,698,3608,551465.54,2431.427066,9199.771204,11029.3108,29177.124790,40206.435590,27645323.52
2,aggressive,16497,698,3608,551465.54,2431.427066,9199.771204,22058.6216,48628.541317,70687.162917,27645323.52
